In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

In [2]:
# ==========================================
# 1. 디렉토리 설정
# ==========================================
DATA_DIR = "./"  
ML_MODEL_DIR = "./model/machine_learning"
TEST_DIR = "./test"

os.makedirs(ML_MODEL_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

print("=" * 50)
print("🚀 [ML] Train / Valid / Test 데이터 로드 및 전처리 시작")
print("=" * 50)

🚀 [ML] Train / Valid / Test 데이터 로드 및 전처리 시작


In [3]:
# ==========================================
# 2. 정형 데이터 로드 및 전처리
# ==========================================
train_path = os.path.join(DATA_DIR, "./data/data/processed/train_20181101_v1.csv")
valid_path = os.path.join(DATA_DIR, "./data/data/processed/valid_20181101_v1.csv")
test_path = os.path.join(DATA_DIR, "./data/data/processed/test_20181101_v1.csv")

train_df = pd.read_csv(train_path)
valid_df = pd.read_csv(valid_path)
test_df = pd.read_csv(test_path)

# 타겟 변수 동적 설정
target_col = "churn" if "churn" in train_df.columns else ("is_churn" if "is_churn" in train_df.columns else train_df.columns[-1])

y_train = train_df[target_col].values
y_valid = valid_df[target_col].values
y_true = test_df[target_col].values  # 원본 앙상블 코드 호환 변수명

# 피처 분리 (타겟 및 식별자 컬럼 제외)
drop_cols = [target_col, "user_id", "userId", "page_sequence", "session_item_sequence", "padded_sequence"]
feature_cols = [col for col in train_df.columns if col not in drop_cols]

X_train = train_df[feature_cols].copy()
X_valid = valid_df[feature_cols].copy()
X_tab_test = test_df[feature_cols].copy()  # 원본 앙상블 코드 호환 변수명

# 범주형 컬럼 전처리 (Train/Valid/Test 범주 코드를 통일하여 Feature Mismatch 방지)
cat_cols = ["gender", "level", "level_last", "last_page"]
for col in cat_cols:
    if col in X_train.columns:
        all_cats = pd.concat([X_train[col], X_valid[col], X_tab_test[col]]).astype(str).astype("category")
        X_train[col] = X_train[col].astype(str).astype("category").cat.set_categories(all_cats.cat.categories).cat.codes
        X_valid[col] = X_valid[col].astype(str).astype("category").cat.set_categories(all_cats.cat.categories).cat.codes
        X_tab_test[col] = X_tab_test[col].astype(str).astype("category").cat.set_categories(all_cats.cat.categories).cat.codes

# 수치형 결측치 정제 (Train과 X_tab_test의 컬럼 개수/순서가 완벽히 일치하도록 보장)
X_train = X_train.apply(pd.to_numeric, errors="coerce").fillna(0.0)
X_valid = X_valid.apply(pd.to_numeric, errors="coerce").fillna(0.0)
X_tab_test = X_tab_test.apply(pd.to_numeric, errors="coerce").fillna(0.0)

# 검증: 학습과 테스트의 피처 개수 및 이름 완전 일치 확인
assert list(X_train.columns) == list(X_tab_test.columns), "⚠️ Train과 X_tab_test의 컬럼 구성이 다릅니다!"
print(f"✅ 피처 검증 완료! (사용된 피처 수: {X_train.shape[1]}개)")

✅ 피처 검증 완료! (사용된 피처 수: 14개)


In [4]:
# ==========================================
# 3. LightGBM 모델 학습 및 저장
# ==========================================
print("\n" + "=" * 50)
print("🚀 [ML] LightGBM 모델 학습 시작")
print("=" * 50)

lgb_model = lgb.LGBMClassifier(
    objective="binary", metric="auc", learning_rate=0.03, 
    max_depth=3, num_leaves=7, n_estimators=500, random_state=42, verbose=-1
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)]
)

lgb_preds = lgb_model.predict_proba(X_tab_test)[:, 1]
print(f"✨ [LightGBM] Test AUC: {roc_auc_score(y_true, lgb_preds):.4f}")

joblib.dump(lgb_model, os.path.join(ML_MODEL_DIR, "lgb_model.pkl"))
np.save(os.path.join(TEST_DIR, "lgb_preds_proba.npy"), lgb_preds)


🚀 [ML] LightGBM 모델 학습 시작
Training until validation scores don't improve for 50 rounds


c:\Users\playdata2\miniconda3\envs\agent\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[100]	valid_0's auc: 0.713018
[200]	valid_0's auc: 0.718753
[300]	valid_0's auc: 0.72069
[400]	valid_0's auc: 0.72088
Early stopping, best iteration is:
[372]	valid_0's auc: 0.721811
✨ [LightGBM] Test AUC: 0.7373


In [5]:
# ==========================================
# 4. CatBoost 모델 학습 및 저장 (.cbm)
# ==========================================
print("\n" + "=" * 50)
print("🚀 [ML] CatBoost 모델 학습 시작")
print("=" * 50)

cb_model = CatBoostClassifier(
    iterations=1000, learning_rate=0.03, depth=4, 
    eval_metric="AUC", random_seed=42, verbose=100
)
cb_model.fit(
    X_train, y_train,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=50,
    use_best_model=True
)

cb_preds = cb_model.predict_proba(X_tab_test)[:, 1]
print(f"✨ [CatBoost] Test AUC: {roc_auc_score(y_true, cb_preds):.4f}")

cb_model.save_model(os.path.join(ML_MODEL_DIR, "catboost_model.cbm"))
np.save(os.path.join(TEST_DIR, "cb_preds_proba.npy"), cb_preds)


🚀 [ML] CatBoost 모델 학습 시작
0:	test: 0.5887764	best: 0.5887764 (0)	total: 169ms	remaining: 2m 48s
100:	test: 0.7000812	best: 0.7000812 (100)	total: 734ms	remaining: 6.53s
200:	test: 0.7099965	best: 0.7102314 (194)	total: 1.44s	remaining: 5.72s
300:	test: 0.7125993	best: 0.7130567 (275)	total: 2.04s	remaining: 4.74s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7130566802
bestIteration = 275

Shrink model to first 276 iterations.
✨ [CatBoost] Test AUC: 0.7337


In [6]:
# ==========================================
# 5. Random Forest 모델 학습 및 저장
# ==========================================
print("\n" + "=" * 50)
print("🚀 [ML] Random Forest 모델 학습 시작")
print("=" * 50)

rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_split=10, 
    min_samples_leaf=5, random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict_proba(X_tab_test)[:, 1]
print(f"✨ [Random Forest] Test AUC: {roc_auc_score(y_true, rf_preds):.4f}")

joblib.dump(rf_model, os.path.join(ML_MODEL_DIR, "rf_model.pkl"))
np.save(os.path.join(TEST_DIR, "rf_preds_proba.npy"), rf_preds)


🚀 [ML] Random Forest 모델 학습 시작
✨ [Random Forest] Test AUC: 0.7259


In [7]:
# ==========================================
# 6. Logistic Regression 모델 학습 및 저장 (Pipeline)
# ==========================================
print("\n" + "=" * 50)
print("🚀 [ML] Logistic Regression 모델 학습 시작")
print("=" * 50)

# 스케일러와 분류기를 Pipeline으로 묶어서 저장하므로 원본 스캔 코드에서 predict_proba 바로 호출 가능
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, C=1.0))
])
lr_pipeline.fit(X_train, y_train)

lr_preds = lr_pipeline.predict_proba(X_tab_test)[:, 1]
print(f"✨ [Logistic Regression] Test AUC: {roc_auc_score(y_true, lr_preds):.4f}")

joblib.dump(lr_pipeline, os.path.join(ML_MODEL_DIR, "lr_model.pkl"))
np.save(os.path.join(TEST_DIR, "lr_preds_proba.npy"), lr_preds)

print("\n🎉 모든 모델 학습 및 저장 완료! 공유해주신 원본 앙상블 스캔 코드와 완벽히 호환됩니다.")


🚀 [ML] Logistic Regression 모델 학습 시작
✨ [Logistic Regression] Test AUC: 0.7226

🎉 모든 모델 학습 및 저장 완료! 공유해주신 원본 앙상블 스캔 코드와 완벽히 호환됩니다.


In [8]:
# ==========================================
# 6. AdaBoost 모델 학습 및 저장
# ==========================================
print("\n" + "=" * 50)
print("🚀 [ML] AdaBoost 모델 학습 시작")
print("=" * 50)

ab_model = AdaBoostClassifier(
    n_estimators=100, learning_rate=1.0, random_state=42
)
ab_model.fit(X_train, y_train)

ab_preds = ab_model.predict_proba(X_tab_test)[:, 1]
print(f"✨ [AdaBoost] Test AUC: {roc_auc_score(y_true, ab_preds):.4f}")

joblib.dump(ab_model, os.path.join(ML_MODEL_DIR, "ab_model.pkl"))
np.save(os.path.join(TEST_DIR, "ab_preds_proba.npy"), ab_preds)


🚀 [ML] AdaBoost 모델 학습 시작
✨ [AdaBoost] Test AUC: 0.7292
